# GeoSR-4 — EDSR with heteroscedastic uncertainty (Colab GPU)

Phase 6 (PRD section 38-39). Single-pass uncertainty head (mean + log-variance, Gaussian NLL loss) instead of the PRD's suggested 5x Monte Carlo ensemble -- much cheaper on a Colab-T4 budget. See `decisions.md` D016 for why, and for a real gradient-explosion bug this loss caused during local smoke-testing at too-high a learning rate (fixed: confirmed stable at lr=1e-4, added gradient clipping as a safety net regardless).

Colab's free tier is typically a T4 -- this notebook uses `--amp` (mixed precision, D035) so its Tensor Cores actually get used. Watch the first few loss values when training starts: if you see `nan` or `inf`, stop and report it (this loss has a known instability history at the wrong settings).

**Before running:** Runtime → Change runtime type → GPU.

In [ ]:
!nvidia-smi

## 1. Clone the repo and install dependencies

In [ ]:
!git clone https://github.com/Vijay6923/GeoSR-4.git
%cd GeoSR-4
!pip install -q rasterio huggingface_hub scikit-image

## 2. Download the dataset (cross-sensor split only, ~2.1 GB)

In [ ]:
from huggingface_hub import hf_hub_download
import zipfile, os

zip_path = hf_hub_download(
    repo_id="isp-uv-es/SEN2NAIP",
    repo_type="dataset",
    filename="cross-sensor/cross-sensor.zip",
    local_dir="ml/datasets/raw/sen2naip",
)

extract_dir = "ml/datasets/raw/sen2naip/cross-sensor/extracted"
os.makedirs(extract_dir, exist_ok=True)
with zipfile.ZipFile(zip_path) as z:
    z.extractall(extract_dir)

print("extracted to", extract_dir)

## 3. Train
Same EDSR-baseline config (16 blocks, 64 channels) as Phase 3, but out_channels=8 (mean + log-variance) and Gaussian NLL loss instead of L1.

In [ ]:
!python ml/training/train_edsr_uncertainty.py \
  --epochs 20 \
  --batch-size 16 \
  --n-blocks 16 \
  --n-channels 64 \
  --lr 1e-4 \
  --amp \
  --checkpoint-dir experiments/edsr_uncertainty \
  --log-every 20

## 4. Full validation-set evaluation (real numbers, not the 50-sample per-epoch estimate)
Reports PSNR/SSIM/SAM/ERGAS on the mean prediction, plus the calibration number -- correlation between predicted uncertainty and actual error. Positive and >0.3-0.4 means the uncertainty map is actually informative, not just noise (D034).

In [ ]:
!python ml/evaluation/evaluate_checkpoint.py \
  --checkpoint experiments/edsr_uncertainty/edsr_unc_epoch19.pt \
  --model-type edsr --uncertainty --n-blocks 16 --n-channels 64

## 5. Download the checkpoint

In [ ]:
from google.colab import files
files.download('experiments/edsr_uncertainty/edsr_unc_epoch19.pt')